New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import sys, os
repo = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
from ai_data_scientist import project_manager as pm, lifecycle, language_router
handle = pm.resolve_project('kaggle-v020-kaggle-exp-15-dots')
pm.ensure_notebook(handle)
lifecycle.register_run('v020-exp-15', handle.notebook_path)
print({'notebook': str(handle.notebook_path), 'language': language_router.detect_language('刺激条件と反応時間・判断の関係を調べてください。'), 'status': lifecycle.get_run_status('v020-exp-15').state})

{'notebook': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-15-dots/notebooks/kaggle-v020-kaggle-exp-15-dots.ipynb', 'language': 'ja', 'status': 'running'}


In [2]:
import nbformat
setup_code = '''# 初期化・実行管理（このセルから順番に再実行可能）
from pathlib import Path
import os, sys, json, hashlib, io, contextlib, inspect, re
from datetime import datetime, timezone
from dataclasses import asdict
repo = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(repo / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(repo / 'benchmark/projects')
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from ai_data_scientist import project_manager as pm, lifecycle, language_router
run_id = 'v020-exp-15'
handle = pm.resolve_project('kaggle-v020-kaggle-exp-15-dots')
pm.ensure_notebook(handle)
data_dir = pm.ensure_data_dir(handle)
lifecycle.register_run(run_id, handle.notebook_path)
phase_log = []
def event(phase, action):
    phase_log.append(dict(time=datetime.now(timezone.utc).isoformat(), phase=phase, action=action))
def start(label):
    if lifecycle.is_cancel_requested(run_id):
        raise RuntimeError('実行キャンセルが要求されました')
    lifecycle.mark_execution_start(run_id); event(label, 'execution_start')
def end(label):
    lifecycle.mark_execution_end(run_id); event(label, 'execution_end')
def write(mutation):
    lifecycle.mark_write_start(run_id); event('notebook', 'write_start')
    try:
        return pm.enqueue_write(handle, mutation)
    finally:
        lifecycle.mark_write_end(run_id); event('notebook', 'write_end')
def add_cells(*cells):
    write(lambda nb: nb.cells.extend(cells))
def md(text):
    return nbformat.v4.new_markdown_cell(text)
def code(text, cell_id):
    return nbformat.v4.new_code_cell(text, id=cell_id)
print({'language': language_router.detect_language('刺激条件と反応時間・判断を分析してください'), 'run_id': run_id, 'status': asdict(lifecycle.get_run_status(run_id))})
'''
api_code = '''# Kaggle SDKで公開データ検索（秘密情報は出力しない）
start('kaggle_acquisition')
queries = ['dots perception experiment', 'dots', 'seaborn dots']
kaggle_log = []
candidates = []
api = None
try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
    kaggle_log.append({'step': 'authenticate', 'success': True})
    for query in queries:
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                results = list(api.dataset_list(search=query))
            refs = [{'ref': str(r.ref), 'title': str(r.title)} for r in results]
            kaggle_log.append({'step': 'dataset_list', 'search': query, 'results': refs})
            candidates.extend(refs)
        except Exception as exc:
            kaggle_log.append({'step': 'dataset_list', 'search': query, 'error_type': type(exc).__name__, 'http_status': getattr(getattr(exc, 'response', None), 'status_code', None)})
except Exception as exc:
    kaggle_log.append({'step': 'authenticate', 'success': False, 'error_type': type(exc).__name__, 'reason': '認証またはSDK初期化失敗。秘密を含む可能性がある例外本文は保存しない。'})
file_inventory = []
if api is not None and any(x.get('success') for x in kaggle_log):
    seen = set()
    for candidate in candidates:
        ref = candidate['ref']
        if ref in seen: continue
        seen.add(ref)
        if len(seen) > 12: break
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                listing = api.dataset_list_files(ref)
            names = [f.name for f in listing.files]
            file_inventory.append({'ref': ref, 'title': candidate['title'], 'files': names})
        except Exception as exc:
            kaggle_log.append({'step': 'dataset_list_files', 'ref': ref, 'error_type': type(exc).__name__})
print(json.dumps({'kaggle_log': kaggle_log, 'file_inventory': file_inventory}, ensure_ascii=False, indent=2))
end('kaggle_acquisition')
'''
lifecycle.mark_write_start('v020-exp-15')
def initialize(nb):
    nb.cells = [nbformat.v4.new_markdown_cell('# Dots知覚実験：刺激条件と反応時間・判断\n\nrun_id: `v020-exp-15`。Kaggle検索と出典検証を先に実施し、集約度と独立単位を確認してから分析する。認証情報は出力・保存しない。'), nbformat.v4.new_code_cell(setup_code, id='setup'), nbformat.v4.new_code_cell(api_code, id='kaggle-search')]
pm.enqueue_write(handle, initialize)
lifecycle.mark_write_end('v020-exp-15')
print('対象Notebookに初期化・Kaggle検索セルを保存しました。')

対象Notebookに初期化・Kaggle検索セルを保存しました。


In [7]:
acquire_code = '''# 対応候補のファイル確認・取得・来歴・一次的構造確認
start('download')
from urllib.request import urlopen, Request
import zipfile
from ai_data_scientist import ingestion, eda
slug = 'daryasikerina/seaborn-dots-dataset'
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    listing = api.dataset_list_files(slug)
files = [f.name for f in listing.files]
print({'owner/dataset': slug, 'files': files})
csv_files = [f for f in files if f.lower().endswith('.csv') and 'dots' in f.lower()]
if len(csv_files) != 1:
    raise ValueError('対応CSVを一意に同定できません: ファイル一覧を再確認してください')
filename = csv_files[0]
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    api.dataset_download_file(slug, filename, path=str(data_dir), force=True, quiet=True)
local_path = data_dir / Path(filename).name
if not local_path.exists():
    archive_path = data_dir / (Path(filename).name + '.zip')
    with zipfile.ZipFile(archive_path) as archive:
        local_path.write_bytes(archive.read(filename))
raw = local_path.read_bytes()
provenance = {'platform': 'Kaggle', 'owner_dataset_slug': slug, 'filename': filename, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(raw).hexdigest(), 'fallback': False, 'search_url': 'https://www.kaggle.com/datasets?search=dots+perception+experiment'}
(data_dir / 'provenance.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
loaded = ingestion.ingest(ingestion.SourceSpec('csv', str(local_path)), row_limit=100000)
df = loaded.dataframe
assert not loaded.truncated
print(json.dumps(provenance, ensure_ascii=False, indent=2))
print('shape:', df.shape); display(df.head(12)); print(df.dtypes.to_string())
print('unique:', {c: df[c].nunique() for c in df}); print('ranges:'); display(df.describe(include='all'))
print('SDK metadata signature:', inspect.signature(api.dataset_metadata))
print('EDA:', eda.explore(df))
source_docs = {}
for label, url in {'seaborn_readme': 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/README.md', 'seaborn_example': 'https://seaborn.pydata.org/examples/errorband_lineplots.html'}.items():
    try:
        with urlopen(Request(url, headers={'User-Agent': 'jupytermind-analysis'}), timeout=30) as response:
            text = response.read().decode('utf-8')
        source_docs[label] = {'url': url, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'text': text}
        (data_dir / (label + '.txt')).write_text(text, encoding='utf-8')
        print(label, re.sub('<[^>]+>', ' ', text)[-6000:] if label == 'seaborn_example' else text)
    except Exception as exc:
        source_docs[label] = {'url': url, 'error_type': type(exc).__name__}
        print({'reference_fetch_failure': label, 'type': type(exc).__name__})
end('download')
'''
add_cells(md('## データ選定\n\n一般のdots検索はゲーム・画像等が混在するため、`seaborn dots`で見つかった`daryasikerina/seaborn-dots-dataset`のCSVを優先する。対応CSVの内容を確認し、不一致なら分析は開始しない。Kaggle API認証・検索に成功しているため、現段階でフォールバックしない。'), code(acquire_code, 'acquisition'))

In [10]:
definition_source_code = '''# 原実験とKaggleメタデータの照合
start('source_verification')
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    api.dataset_metadata(slug, path=str(data_dir))
metadata_paths = list(data_dir.glob('*metadata*.json'))
for p in metadata_paths:
    meta = json.loads(p.read_text(encoding='utf-8'))
    print('Kaggle metadata:', json.dumps({k: meta.get(k) for k in ['id', 'title', 'description', 'licenses']}, ensure_ascii=False))
url = 'https://shadlenlab.columbia.edu/resources/RoitmanDataCode.html'
with urlopen(Request(url, headers={'User-Agent': 'jupytermind-analysis'}), timeout=30) as response:
    primary_text = response.read().decode('utf-8', errors='replace')
source_docs['original_experiment'] = {'url': url, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'text': primary_text}
(data_dir / 'original_experiment.html').write_text(primary_text, encoding='utf-8')
print(re.sub('<[^>]+>', ' ', primary_text)[:18000])
print('links:', re.findall(r'href=[\"\x27]([^\"\x27]+)', primary_text))
print('cell keys duplicate:', int(df.duplicated(['align', 'choice', 'coherence', 'time']).sum()))
print(df.groupby(['align','choice','coherence']).agg(n_bins=('time','size'), minimum=('time','min'), maximum=('time','max')).to_string())
end('source_verification')
'''
add_cells(code(definition_source_code, 'source-verification'))

In [18]:
manifest_code = '''# データ定義・分析前提manifestと意味的品質検査
start('manifests_quality')
from ai_data_scientist.data_definition import FieldValue, build_manifest
from ai_data_scientist.analysis_assumptions import Assumption, AnalysisAssumptionManifest, check_manifest
from ai_data_scientist import data_quality, cleaning
paper_ref = 'Roitman & Shadlen (2002), DOI:10.1523/JNEUROSCI.22-21-09475.2002; Europe PMC PMID12417672'
V = FieldValue
variables = {
 'align': {'meaning': V('dots=刺激開始、sacc=眼球運動開始への時間整列', 'inferred', '列名・時間軸・原実験とSeaborn利用例'), 'unit': V('カテゴリー','verified','CSV'), 'levels': V(['dots','sacc'],'verified','CSV')},
 'choice': {'meaning': V('二つの眼球運動選択先T1/T2。各行の二値判断ではなく選択条件', 'inferred', paper_ref), 'target_mapping': V('T1の受容野内/正答への対応はCSV辞書未確認', 'unknown'), 'unit': V('カテゴリー','verified','CSV')},
 'time': {'meaning': V('整列イベントからの相対時間であり、試行反応時間ではない', 'inferred','負値、20刻み、align別曲線'), 'unit': V('msと推測。CSVに単位辞書なし', 'inferred', paper_ref), 'reaction_time_available': V(False,'verified','CSVの全5列を確認')},
 'coherence': {'meaning': V('一貫した方向に動くランダムドットの割合（刺激強度）', 'inferred',paper_ref), 'unit': V('%と推測。CSV辞書なし','inferred'), 'levels': V([0,3.2,6.4,12.8,25.6,51.2],'verified','CSV')},
 'firing_rate': {'meaning': V('条件・時間別に要約された神経発火率', 'inferred', 'CSV構造と原実験'), 'unit': V('spikes/sと推測。集約・平滑化の方法は不明','inferred',paper_ref), 'aggregation': V('ニューロン/試行数、重み、平滑化幅が不明','unknown')}
}
definition = build_manifest(
 source={k: V(v,'verified','Kaggle取得時の計測') for k,v in provenance.items()} | {'license': V('unknown','unknown','Kaggle metadata: licenses=unknown'), 'canonical_identity': V('Seaborn原版との完全同一性未確認','unknown')},
 dataset_scope={'row_unit': V('align × choice × coherence × timeに一意な曲線上の点','verified','複合キー検査'), 'experimental_unit': V('原実験はアカゲザルの運動方向判断試行とLIP神経記録','verified',paper_ref), 'independent_units_in_csv': V('個体・ニューロン・試行IDと独立数が欠落','unknown','全列検査'), 'csv_n': V(len(df),'verified','CSV')}, variables=variables,
 transformations=('原CSVをそのまま保持。欠測・複合キー重複を検査後、共通時間窓に限定した要約を作成',))
print('unknown_fields:', [(p,asdict(v)) for p,v in definition.unresolved_fields()])
inferred = [(f'variables.{c}.{k}',asdict(v)) for c,fields in variables.items() for k,v in fields.items() if v.status == 'inferred']
print('inferred_fields (別途列挙):', inferred)
assumptions = AnalysisAssumptionManifest(
 analysis_scope={'outcome':'条件別発火率曲線の記述。反応時間、正答率、選択確率は推定不能','primary_window':[100,280],'baseline_window':[-80,-20],'response_window':[-100,0]},
 causal_scope='descriptive', sampling={'n':len(df),'seed':20261002},
 assumptions=(
 Assumption('curve_not_trial','行を独立試行とみなさず条件曲線として集約する','tested',impact_if_false='独立反復数を過大評価する',conclusion_critical=True),
 Assumption('common_support','同じalign内で全条件に共通する時間窓だけを比較する','tested',impact_if_false='条件ごとの時間範囲差が比較を歪める',conclusion_critical=True),
 Assumption('unit_mapping','時間ms・発火率spikes/s・coherence%への対応','assumed',impact_if_false='物理単位での解釈・換算が変わる。数値単位の記述に限定',conclusion_critical=True),
 Assumption('no_identification','観察曲線から因果効果・個体集団への有意差を推定できる','rejected',impact_if_false='原試行データがあれば再設計可能',conclusion_critical=False),
 Assumption('window_choice','100..280を主窓とすることが実質的な結論を左右しない','assumed',impact_if_false='窓の感度分析を行い頑健と断言しない',conclusion_critical=True)))
findings = check_manifest(assumptions)
print('assumption_findings:', [asdict(f) for f in findings])
expected_columns = {'align','choice','time','coherence','firing_rate'}
assert set(df.columns) == expected_columns
schema = {'align':{'allowed':['dots','sacc'],'not_null':True}, 'choice':{'allowed':['T1','T2'],'not_null':True}, 'time':{'min':-1000,'max':1500,'not_null':True}, 'coherence':{'allowed':[0.,3.2,6.4,12.8,25.6,51.2],'min':0,'max':100,'not_null':True}, 'firing_rate':{'min':0,'not_null':True}}
anomalies = data_quality.detect_anomalies(df,schema)
nonfinite = {c:int((~np.isfinite(df[c].to_numpy())).sum()) for c in ['time','coherence','firing_rate']}
key_duplicates = int(df.duplicated(['align','choice','coherence','time']).sum())
spacing_bad = []
for key,g in df.groupby(['align','choice','coherence']):
    steps = np.diff(np.sort(g.time.to_numpy()))
    if not np.all(steps == 20): spacing_bad.append(str(key))
print('semantic_anomalies:', [asdict(a) for a in anomalies]); print('nonfinite:',nonfinite,'key_duplicates:',key_duplicates,'spacing_bad:',spacing_bad)
assert not anomalies and not key_duplicates and not any(nonfinite.values()) and not spacing_bad
clean_report = cleaning.clean_dataset(df,'drop_duplicates')
print('cleaning_impact:', {'rows_before':clean_report.rows_before,'rows_after':clean_report.rows_after,'rows_removed':clean_report.rows_removed,'columns_affected':clean_report.columns_affected})
assert clean_report.rows_removed == 0
support = df.groupby(['align','choice','coherence']).time.agg(['min','max','size'])
print('common_support:', support.groupby(level='align').agg({'min':'max','max':'min'}).to_dict('index'))
print('IMPORTANT: timeは反応時間ではない。848行は848試行ではない。T1/T2行数は選択確率ではない。')
for name,obj in {'data_definition_manifest':asdict(definition),'analysis_assumptions_manifest':asdict(assumptions),'semantic_quality':{'schema':schema,'anomalies':[asdict(a) for a in anomalies],'key_duplicates':key_duplicates,'nonfinite':nonfinite,'spacing_bad':spacing_bad}}.items():
    (data_dir / (name+'.json')).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown(f'DATA_ROWS={len(df)}; UNIQUE_CURVES={len(support)}; REACTION_TIME_AVAILABLE=False; CHOICE_PROBABILITY_AVAILABLE=False'))
end('manifests_quality')
'''
analysis_code = '''# 初回分析：共通サポート上の効果量（独立試行の検定は行わない）
start('initial_analysis')
from scipy.stats import spearmanr, theilslopes
coherences = sorted(df.coherence.unique())
def selected(align,lo,hi):
    out = df[(df['align']==align)&df.time.between(lo,hi)].copy()
    counts = out.groupby(['choice','coherence']).size()
    assert len(counts)==12 and counts.nunique()==1, '比較窓が共通サポートを外れています'
    return out
ramp = selected('dots',100,280)
baseline = selected('dots',-80,-20)
response = selected('sacc',-100,0)
base_means = baseline.groupby(['choice','coherence']).firing_rate.mean()
rows = []
for (choice,coherence),g in ramp.groupby(['choice','coherence']):
    g = g.sort_values('time')
    slope = float(np.polyfit(g.time,g.firing_rate,1)[0])*100
    rows.append({'choice':choice,'coherence':float(coherence),'n_time_points':len(g),'mean_rate':g.firing_rate.mean(),'baseline_mean':base_means.loc[(choice,coherence)],'baseline_change':g.firing_rate.mean()-base_means.loc[(choice,coherence)],'slope_per_100_time_units':slope})
summary = pd.DataFrame(rows)
pair = ramp.pivot(index=['coherence','time'],columns='choice',values='firing_rate')
pair['T1_minus_T2'] = pair.T1-pair.T2
gap_by_coh = pair.groupby(level='coherence').T1_minus_T2.mean()
resp_pair = response.pivot(index=['coherence','time'],columns='choice',values='firing_rate')
resp_pair['T1_minus_T2'] = resp_pair.T1-resp_pair.T2
response_summary = resp_pair.groupby(level='coherence')[['T1','T2','T1_minus_T2']].mean()
print('条件別初回要約:'); display(summary.round(6))
print('共通窓100..280のT1−T2:'); display(gap_by_coh.to_frame().round(6))
print('sacc窓−100..0（同じ元データの別整列であり独立再現ではない）:'); display(response_summary.round(6))
primary_effect = float(gap_by_coh.loc[51.2]-gap_by_coh.loc[0.0])
slopes = summary.pivot(index='coherence',columns='choice',values='slope_per_100_time_units')
rank_stats = {ch:float(spearmanr(slopes.index.to_numpy(),slopes[ch].to_numpy()).statistic) for ch in ['T1','T2']}
print('coherenceと窓内傾きのSpearman rho（6設計水準の記述量、p値なし）:',rank_stats)
print('理由: 時間点は自己相関し、独立個体/ニューロン/試行数が不明。行単位t検定、ANOVA、ロジスティック回帰、通常の相関p値・CI・ブートストラップを使わない。')
print('母集団SEやCIは識別不能。時間点のばらつきをその代用にしない。')
summary.to_csv(data_dir/'initial_summary.csv',index=False)
response_summary.to_csv(data_dir/'response_summary.csv')
display(Markdown(f'PRIMARY_GAP_INTERACTION={primary_effect:.9f}; LOW_COH_GAP={gap_by_coh.loc[0.0]:.9f}; HIGH_COH_GAP={gap_by_coh.loc[51.2]:.9f}; T1_SLOPE_RHO={rank_stats["T1"]:.9f}; T2_SLOPE_RHO={rank_stats["T2"]:.9f}'))
end('initial_analysis')
'''
chart_code = '''# 図1：整列・選択先・刺激条件別の全曲線／図2：共通窓で比較
start('figures')
import matplotlib.pyplot as plt
import warnings
from matplotlib import font_manager
from ai_data_scientist.visualization import render_chart, build_image_output
chart_specs = {}
def show_figure(fig,name,title,xlabel,ylabel,legend):
    buf = io.BytesIO()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        fig.savefig(buf,format='png',dpi=130,bbox_inches='tight')
    glyph_warnings = [str(w.message) for w in caught if 'Glyph' in str(w.message)]
    if glyph_warnings: raise RuntimeError(str(glyph_warnings))
    png = buf.getvalue()
    (data_dir/(name+'.png')).write_bytes(png)
    display(build_image_output(png)['data'],raw=True)
    plt.close(fig)
    chart_specs[name]={'missing_glyphs':[],'title':title,'xlabel':xlabel,'ylabel':ylabel,'legend':legend,'glyph_check':'savefig warning capture; all labels ASCII','png_bytes':len(png)}
colors = plt.cm.viridis(np.linspace(0,1,len(coherences)))
fig,axes=plt.subplots(2,2,figsize=(12,8),sharey=True)
for i,align in enumerate(['dots','sacc']):
    for j,choice in enumerate(['T1','T2']):
        ax=axes[i,j]
        for coh,col in zip(coherences,colors):
            g=df[(df['align']==align)&(df.choice==choice)&(df.coherence==coh)].sort_values('time')
            ax.plot(g.time,g.firing_rate,color=col,label=f'{coh:g}')
        ax.axvline(0,color='black',linestyle=':',linewidth=1)
        ax.axvspan(100 if align=='dots' else -100,280 if align=='dots' else 0,color='grey',alpha=.12)
        ax.set(title=f'{align} alignment / {choice}',xlabel='Relative time (CSV units; ms inferred)',ylabel='Firing rate (CSV units; spikes/s inferred)')
        ax.legend(title='Coherence (CSV units)',fontsize=8,ncol=2)
        ax.grid(alpha=.15)
fig.suptitle('Condition-averaged neural curves, not trial reaction times',fontsize=13)
fig.tight_layout()
show_figure(fig,'trajectories','Condition-averaged neural curves','Relative time (CSV units)','Firing rate (CSV units)','Coherence: six levels; panels: align/choice')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for choice,marker in [('T1','o'),('T2','s')]:
    g=summary[summary.choice==choice]
    axes[0].plot(g.coherence,g.slope_per_100_time_units,marker=marker,label=choice)
axes[0].axhline(0,color='grey',linewidth=.8)
axes[0].set(title='Descriptive ramp slopes: dots 100..280',xlabel='Coherence (CSV units; % inferred)',ylabel='Rate change per 100 time units')
axes[0].legend(title='Choice condition')
axes[1].plot(gap_by_coh.index,gap_by_coh.values,marker='o',label='dots: 100..280')
axes[1].plot(response_summary.index,response_summary.T1_minus_T2,marker='s',label='sacc: -100..0')
axes[1].set(title='Matched-window choice contrast',xlabel='Coherence (CSV units; % inferred)',ylabel='Mean T1 - T2 firing rate (CSV units)')
axes[1].legend(title='Alignment/window')
fig.tight_layout()
show_figure(fig,'matched_contrasts','Matched-window contrasts','Coherence (CSV units)','Slope / choice contrast (CSV units)','Choice / alignment-window')
# Skill標準のrender_chartも、単一の記述図に実際に使用する。
chart_df=gap_by_coh.reset_index()
png=render_chart(chart_df,kind='line',x='coherence',y='T1_minus_T2',title='Choice contrast on common support: dots 100..280',xlabel='Coherence (CSV units)',ylabel='T1 - T2 firing rate (CSV units)')
(data_dir/'skill_gap_chart.png').write_bytes(png)
display(build_image_output(png)['data'],raw=True)
chart_specs['skill_gap_chart']={'missing_glyphs':[],'title':'Choice contrast on common support','xlabel':'Coherence (CSV units)','ylabel':'T1 - T2 firing rate (CSV units)','legend':'T1_minus_T2','glyph_check':'ASCII labels'}
print('図の目視確認項目: 条件ごとの線・凡例、0基準、灰色比較窓、単位の推定表示、軸の同一尺度。曲線端点を反応時間とは呼ばない。')
print('chart_specs:',chart_specs)
end('figures')
'''
plan_md = '''## 変数の意味・実験単位と分析計画（統計計算前に固定）

Seabornの出典READMEはdotsをRoitman/Shadlenの原実験ページに結びつける。原論文はアカゲザルのランダムドット運動方向判断・眼球運動反応とLIP神経発火を扱う（Roitman & Shadlen, 2002、DOI:10.1523/JNEUROSCI.22-21-09475.2002、PMID12417672）。Kaggle掲載説明は空でライセンスunknown。原実験公開ページは接続タイムアウトだったため、Europe PMCの原論文抄録とSeabornの利用例を照合した。原版CSVとの完全同一性・加工経路は検証していない。

|列|扱い|確信度と制約|
|---|---|---|
|align|dots:刺激、sacc:眼球運動への時間整列と推定|CSVに定義辞書がないためinferred|
|choice|T1/T2の選択条件|正答/誤答、左/右、受容野への対応はunknown|
|time|イベント基準の相対時間|負値もあり、試行反応時間ではない。msはinferred|
|coherence|運動の一貫性（刺激強度）|6水準。百分率はinferred|
|firing_rate|条件別神経発火率の曲線値|spikes/sと集約方法は未検証|

CSVの記録単位は条件×時間点である。原実験の試行・個体・ニューロンの独立数は復元できない。`time`から反応時間平均や分布を、`choice`行数から選択確率・正答率を算出しない。反応時間と正確性の関係は原論文の報告と今回の数値分析を明確に区別する。

**主分析**は共通サポート内でのdots時間100..280の線形傾き（100時間単位あたり）、T1−T2の同時刻対応差、coherence 51.2−0での選択差の変化。刺激前−80..−20の平均も併記。sacc −100..0の比較は同じ記録の別整列であり独立再現ではない。coherenceと傾きのSpearman係数は6設計水準の記述量に限定する。単位未確認により数値単位を基本とする。

**図表**は全曲線の2×2ファセット、共通窓の傾きと選択差、条件別要約表。独立反復がないので有意差検定・母集団CI・行単位ブートストラップはしない。窓幅・開始時刻・平均/中央値と頑健傾きの感度を調べ、切り取りによる結論変化を確認する。

**不適用**：独立参照データがないため`validate_anomalies`、`compare_datasets`は不適用。同じCSVの別整列は独立参照ではない。試行IDがないため混合モデル・反応時間モデル・判断ロジスティックモデルは不適用。相関p値を自動出力する`stats_analysis.correlation`も独立標本条件を満たさず使わない。`mcp_gateway.run_and_record`はホストのMCPクライアントを要求し、実行中カーネル内で自分を再実行すると待ち合わせになるため使わない。全分析コードの実行はホストのJupyter MCP `execute_cell`。ソース/Markdown/監査メタデータは`enqueue_write`、実行結果保存はJupyter MCP自身が担当する。ゲートウェイ使用と偽って記録しない。
'''
add_cells(md(plan_md),code(manifest_code,'manifests'),code(analysis_code,'initial-analysis'),code(chart_code,'figures'))
print('計画・manifest・初回分析・図表セルを保存しました。')

計画・manifest・初回分析・図表セルを保存しました。


In [22]:
request1 = '共通時間窓で刺激強度が高いほどT1−T2の発火率差が大きいという初回結果について、窓の開始・終了時刻、平均と中央値、刺激前ベースライン補正を変えてください。OLSとTheil–Senの傾きも比較し、効果の方向と大きさのどこまで安定しているかを区別してください。試行レベルの有意差や反応時間効果には読み替えないでください。'
iteration1_code = '''# 反復1：主効果の窓・要約・ベースライン感度
start('iteration_1')
from ai_data_scientist.sensitivity import SensitivityPlan, run_sensitivity
request_1 = ''' + repr(request1) + '''
print('追加依頼原文:',request_1)
baseline_pair = baseline.pivot(index=['coherence','time'],columns='choice',values='firing_rate')
baseline_pair['gap']=baseline_pair.T1-baseline_pair.T2
baseline_gap=baseline_pair.groupby(level='coherence').gap.mean()
def gap_effect(lo,hi,reducer,correction):
    g=selected('dots',lo,hi).pivot(index=['coherence','time'],columns='choice',values='firing_rate')
    g['gap']=g.T1-g.T2
    stats=g.groupby(level='coherence').gap.agg(reducer)
    if correction=='subtract': stats=stats-baseline_gap
    return float(stats.loc[51.2]-stats.loc[0.0])
sensitivity_plan=SensitivityPlan(parameter_grid={'lo':[100,140,180],'hi':[280,260,240],'reducer':['mean','median'],'correction':['raw','subtract']},max_runs=36)
sensitivity_report=run_sensitivity(sensitivity_plan,gap_effect,stability_tolerance=0.20)
sensitivity_table=pd.DataFrame([{**r.specification,'effect':r.value} for r in sensitivity_report.results])
assert len(sensitivity_table)==36
slope_rows=[]
for lo,hi in [(100,280),(140,280),(180,280)]:
    for method in ['OLS','Theil-Sen']:
        for (choice,coh),g in selected('dots',lo,hi).groupby(['choice','coherence']):
            slope=(np.polyfit(g.time,g.firing_rate,1)[0] if method=='OLS' else theilslopes(g.firing_rate,g.time).slope)*100
            slope_rows.append({'lo':lo,'hi':hi,'method':method,'choice':choice,'coherence':coh,'slope':float(slope)})
slope_sensitivity=pd.DataFrame(slope_rows)
slope_extremes=slope_sensitivity[slope_sensitivity.coherence.isin([0,51.2])].pivot(index=['lo','hi','method','choice'],columns='coherence',values='slope')
slope_extremes['high_minus_low']=slope_extremes[51.2]-slope_extremes[0.0]
print('36仕様の効果量:'); display(sensitivity_table.round(6))
print('傾きの感度（CSV rate単位/100 time単位）:'); display(slope_extremes.round(6))
print('stable:',sensitivity_report.stable,'max_relative_deviation:',sensitivity_report.max_relative_deviation,'tolerance:',.20)
print('全仕様が正:',bool((sensitivity_table.effect>0).all()),'baseline_correction_effect:',gap_effect(100,280,'mean','subtract'))
print('注意: 最大相対偏差は最初の仕様との差。方向安定と20%以内の大きさ安定は別判定。仕様間の最小最大はCIではない。')
sensitivity_table.to_csv(data_dir/'sensitivity_36_specs.csv',index=False)
slope_sensitivity.to_csv(data_dir/'slope_sensitivity.csv',index=False)
(data_dir/'sensitivity_report.json').write_text(json.dumps(asdict(sensitivity_report),ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown(f'SENSITIVITY_MIN={sensitivity_table.effect.min():.9f}; SENSITIVITY_MAX={sensitivity_table.effect.max():.9f}; SENSITIVITY_STABLE={sensitivity_report.stable}; MAX_RELATIVE_DEVIATION={sensitivity_report.max_relative_deviation:.9f}; ALL_EFFECTS_POSITIVE={bool((sensitivity_table.effect>0).all())}'))
end('iteration_1')
'''
add_cells(md('## 反復依頼履歴\n\n### 反復1：窓とベースラインの感度\n\n選定理由：主窓に共通の刺激誘発波形が含まれること、刺激前のT1/T2差がゼロでないこと、曲線が非線形であることが効果量に実質的な影響を与えうる。初回分析・図1/2を読み直して選んだ。\n\n追加依頼文（原文）：\n\n> '+request1+'\n\n実行セル：`iteration-1`。36仕様を上限内ですべて実行し、仕様範囲をCIとして扱わない。'),code(iteration1_code,'iteration-1'))

In [24]:
request2 = '感度分析では刺激強度によるT1−T2差の変化が一部の仕様で負になりました。高coherenceとゼロcoherenceの選択差の差を、共通サポート上で時間点ごとに表示し、刺激前差を差し引いた系列も併記してください。窓平均・中央値が食い違う原因を時間経過から調べ、全時間帯で同じ方向に働くという結論を維持できるか判断してください。符号交差を反応時間や統計的な効果発現時刻と呼ばないでください。'
iteration2_code = '''# 反復2：窓依存・符号反転の時間的内訳
start('iteration_2')
request_2 = ''' + repr(request2) + '''
print('追加依頼原文:',request_2)
common=selected('dots',-80,280).pivot(index=['coherence','time'],columns='choice',values='firing_rate')
common['gap']=common.T1-common.T2
all_gap=common.gap.unstack('coherence').sort_index()
time_effect=pd.DataFrame({'raw_high_minus_zero_gap':all_gap[51.2]-all_gap[0.]})
base_interaction=float(baseline_gap.loc[51.2]-baseline_gap.loc[0.])
time_effect['baseline_corrected']=time_effect.raw_high_minus_zero_gap-base_interaction
bins=[(100,140),(160,200),(220,240),(260,280)]
block_results=[]
for lo,hi in bins:
    g=time_effect.loc[lo:hi]
    block_results.append({'lo':lo,'hi':hi,'n_time_points':len(g),'raw_mean':g.raw_high_minus_zero_gap.mean(),'corrected_mean':g.baseline_corrected.mean()})
blocks=pd.DataFrame(block_results)
print('時間点ごとの差:'); display(time_effect.round(6))
print('探索的ブロック要約（非重複・全主窓を覆う、独立標本ではない）:'); display(blocks.round(6))
positive_count=int((time_effect.loc[100:280].baseline_corrected>0).sum())
print('主窓中の正の補正済み時間点:',positive_count,'/',len(time_effect.loc[100:280]))
print('交差は曲線の記述であり、潜時や反応時間の推定ではない。平滑化方法が不明なため点ごとの検定も行わない。')
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(time_effect.index,time_effect.raw_high_minus_zero_gap,marker='o',label='Raw choice-gap interaction: 51.2 - 0')
ax.plot(time_effect.index,time_effect.baseline_corrected,marker='s',label='Subtract pre-stimulus interaction')
ax.axhline(0,color='black',linewidth=.8); ax.axvline(0,color='grey',linestyle=':')
ax.axvspan(100,280,color='grey',alpha=.1)
ax.set(title='Time-resolved interaction explains window sensitivity',xlabel='Relative time (CSV units; ms inferred)',ylabel='Difference of T1-T2 gaps (CSV rate units)')
ax.legend(title='Coherence contrast / baseline specification',fontsize=9)
fig.tight_layout()
show_figure(fig,'time_interaction','Time-resolved interaction','Relative time (CSV units)','Difference of choice gaps','Raw / baseline-corrected')
time_effect.to_csv(data_dir/'time_interaction.csv')
blocks.to_csv(data_dir/'exploratory_time_blocks.csv',index=False)
display(Markdown(f'EARLY_CORRECTED_MEAN={blocks.iloc[0].corrected_mean:.9f}; LATE_CORRECTED_MEAN={blocks.iloc[-1].corrected_mean:.9f}; POSITIVE_CORRECTED_BINS={positive_count}; MAIN_WINDOW_BINS={len(time_effect.loc[100:280])}'))
end('iteration_2')
'''
add_cells(md('## 反復1の読み直しと反復2の選定\n\n36仕様では大きさだけでなく符号も変化した。したがって「刺激強度が高いほど選択差が常に大きい」という一般化は取り下げ、時間依存の記述へ修正する。OLSとTheil–Senの高低傾き差は同方向だが、平均レベルの差とは別の量である。証拠：`iteration-1`の全仕様表と出力トークン。残る限界：単位、平滑化、独立反復が不明。\n\n### 反復2：符号反転を時間経過に分解\n\n選定理由：方向まで変わるという反復1の結果が初回結論を実質的に変えたため。窓を都合よく選び直すのではなく、共通サポート全体の系列を表示する。\n\n追加依頼文（原文）：\n\n> '+request2+'\n\n実行セル：`iteration-2`。'),code(iteration2_code,'iteration-2'))

In [26]:
request3 = '時間依存の差は確認できましたが、条件ごとに曲線の長さが違うため、反応時間や判断頻度を誤って読み取る危険が残ります。dotsの終点とsaccの始点、条件別時間点数の関係を調べ、全行を使う不揃いな時間平均と共通窓平均を対比してください。T1/T2の行数比も共通サポートへの切り取りでどう変わるか確認し、CSVから行動反応時間・正答率・選択確率を推定できない理由を具体的に説明してください。'
iteration3_code = '''# 反復3：支持域・擬似判断頻度・反応時間の誤読防止
start('iteration_3')
request_3 = ''' + repr(request3) + '''
print('追加依頼原文:',request_3)
support_table=df.groupby(['align','choice','coherence']).time.agg(['min','max','size']).reset_index()
dots_support=support_table[support_table['align']=='dots'].set_index(['choice','coherence'])
sacc_support=support_table[support_table['align']=='sacc'].set_index(['choice','coherence'])
endpoint_table=pd.DataFrame({'dots_max':dots_support['max'],'sacc_min':sacc_support['min'],'dots_n_points':dots_support['size'],'sacc_n_points':sacc_support['size']})
endpoint_table['sum_of_endpoints']=endpoint_table.dots_max+endpoint_table.sacc_min
print('支持域と整列間の端点関係:'); display(endpoint_table)
raw_means=df[df['align']=='dots'].groupby(['coherence','choice']).firing_rate.mean().unstack('choice')
raw_gap=raw_means.T1-raw_means.T2
estimator_comparison=pd.DataFrame({'unequal_full_curve_gap':raw_gap,'matched_100_280_gap':gap_by_coh})
print('不揃い平均は異なる時間帯を比較する診断例（採用しない）:'); display(estimator_comparison.round(6))
raw_share=float((df.choice=='T2').mean())
matched_df=pd.concat([selected('dots',-80,280),selected('sacc',-160,300)],ignore_index=True)
matched_share=float((matched_df.choice=='T2').mean())
print('行数比診断:', {'all_rows':len(df),'all_rows_T2_share':raw_share,'common_support_rows':len(matched_df),'common_support_T2_share':matched_share})
print('この行数比は保存された時間ビンの比であり、判断頻度ではない。端点は記録された曲線の範囲であり、試行反応時間の統計量ではない。')
print('全12条件のdots_max+sacc_min:',sorted(endpoint_table.sum_of_endpoints.unique().tolist()))
print('整列間の機械的な端点対応は切り取り規則を示唆するが、具体的な規則・平均RTとの関係は不明。')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for choice,marker in [('T1','o'),('T2','s')]:
    g=dots_support.loc[choice].sort_index()
    axes[0].plot(g.index,g['max'],marker=marker,label=choice)
axes[0].set(title='Stored curve endpoints, NOT reaction times',xlabel='Coherence (CSV units)',ylabel='Last dots-aligned time point (CSV units)')
axes[0].legend(title='Choice condition')
axes[1].plot(estimator_comparison.index,estimator_comparison.unequal_full_curve_gap,marker='s',label='Unequal full curves: diagnostic only')
axes[1].plot(estimator_comparison.index,estimator_comparison.matched_100_280_gap,marker='o',label='Matched window: primary')
axes[1].set(title='Support choice changes apparent level contrast',xlabel='Coherence (CSV units)',ylabel='Mean T1 - T2 firing rate (CSV units)')
axes[1].legend(title='Estimator',fontsize=8)
fig.tight_layout()
show_figure(fig,'support_diagnostic','Support and estimator diagnostics','Coherence (CSV units)','Endpoint / gap (CSV units)','Choice / estimator')
endpoint_table.to_csv(data_dir/'support_endpoints.csv')
estimator_comparison.to_csv(data_dir/'support_estimator_comparison.csv')
display(Markdown(f'RAW_ROW_T2_SHARE={raw_share:.9f}; MATCHED_ROW_T2_SHARE={matched_share:.9f}; ENDPOINT_SUM_VALUES={endpoint_table.sum_of_endpoints.unique().tolist()}; RT_ESTIMABLE=False; ACCURACY_ESTIMABLE=False; CHOICE_PROBABILITY_ESTIMABLE=False'))
end('iteration_3')
'''
probe_code = '''# Jupytermind改善候補の最小再現（外部API/CLI問題とは分離）
start('jupytermind_probes')
from ai_data_scientist import notebook_audit, insight_engine
probe_results={}
probe_root=data_dir/'audit_probes'
stream_handle=pm.resolve_project('stream-evidence',projects_root=probe_root)
pm.ensure_notebook(stream_handle)
def stream_fixture(nb):
    cell=nbformat.v4.new_code_cell('print("STREAM_PROBE=7")',execution_count=1)
    cell.outputs=[nbformat.v4.new_output('stream',name='stdout',text='STREAM_PROBE=7')]
    nb.cells=[cell]
pm.enqueue_write(stream_handle,stream_fixture)
try:
    insight_engine.record_insight(stream_handle,'Stream根拠の記録確認',1,'STREAM_PROBE=7','descriptive',language='ja')
    probe_results['stream_evidence']={'expected':'通常のstdoutを証拠として受け付ける','actual':'accepted','candidate':False}
except insight_engine.EvidenceMissingError as exc:
    probe_results['stream_evidence']={'expected':'通常のstdoutを証拠として受け付ける','actual':type(exc).__name__,'candidate':True,'workaround':'主分析はdisplay(Markdown(...))のtext/plain MIMEに出力して照合'}
visual_handle=pm.resolve_project('missing-chart-metadata',projects_root=probe_root)
pm.ensure_notebook(visual_handle)
def visual_fixture(nb):
    cell=nbformat.v4.new_code_cell('render_chart(...)',execution_count=1)
    out=build_image_output((data_dir/'skill_gap_chart.png').read_bytes())
    out.execution_count=1
    cell.outputs=[out]
    nb.cells=[cell]
pm.enqueue_write(visual_handle,visual_fixture)
visual_probe=notebook_audit.audit_notebook(visual_handle.notebook_path,visual_audit=True)
probe_results['missing_chart_metadata']={'expected':'chartメタデータがない画像セルについて監査不可または欠落を通知','actual':{'ok':visual_probe.ok,'visual_findings':[asdict(f) for f in visual_probe.visual_findings]},'candidate':bool(visual_probe.ok and not visual_probe.visual_findings),'workaround':'実際の描画コードで確認したタイトル/軸/凡例/字形情報を明示的にchartメタデータへ保存'}
print(json.dumps(probe_results,ensure_ascii=False,indent=2))
(data_dir/'jupytermind_probe_log.json').write_text(json.dumps(probe_results,ensure_ascii=False,indent=2),encoding='utf-8')
# 検証用に作った特定ファイル・空ディレクトリだけを削除する。
for h in [stream_handle,visual_handle]:
    h.notebook_path.unlink(); h.notebook_path.parent.rmdir(); h.root.rmdir()
probe_root.rmdir()
display(Markdown('JUPYTERMIND_PROBE_LOG='+json.dumps(probe_results,ensure_ascii=False)))
end('jupytermind_probes')
'''
add_cells(md('## 反復2の読み直しと反復3の選定\n\n時間点別表示では初回窓内の前半と後半で高低coherenceの選択差の差が反転していた。長い窓の平均は後半の大きい差に引っ張られ、中央値は多くの小さい/負の時間点を反映する。このため主窓に限った平均効果と、全時間を通じた単調効果を区別する。証拠：`iteration-2`の時間別表・図。残る限界：符号交差は統計的潜時ではなく、未知の平滑化に依存する。\n\n### 反復3：支持域と行動指標の誤読\n\n選定理由：反応時間・判断との関係という元の目的に対して、支持域の短縮や行数比を行動統計と取り違える未解決リスクが残るため。\n\n追加依頼文（原文）：\n\n> '+request3+'\n\n実行セル：`iteration-3`。'),code(iteration3_code,'iteration-3'),md('## Jupytermind改善候補の再現検査\n\n主分析とは別の小さな一時Notebookを`project_manager`で作成し、出力形式と図監査を独立に検証する。ネットワーク、Kaggle、Copilot CLI、ベンチマークランナーは再現条件に含めない。期待された例外は型・影響を明示し、分析失敗として隠さない。'),code(probe_code,'jupytermind-probes'))

In [29]:
first_pass = {'sha256':provenance['sha256'],'retrieved_at_utc':provenance['retrieved_at_utc'],'primary_effect':primary_effect,'sensitivity_min':float(sensitivity_table.effect.min()),'sensitivity_max':float(sensitivity_table.effect.max()),'max_relative_deviation':sensitivity_report.max_relative_deviation,'early_corrected_mean':float(blocks.iloc[0].corrected_mean),'late_corrected_mean':float(blocks.iloc[-1].corrected_mean),'raw_row_share':raw_share,'matched_row_share':matched_share,'iterations':3}
(data_dir/'first_pass_reference.json').write_text(json.dumps(first_pass,ensure_ascii=False,indent=2),encoding='utf-8')
module_md = '''## 使用したJupytermindモジュール

以下はこのNotebook内で直接importし、直接呼び出したもののみ。間接importを使用済みに数えない。

|モジュール|直接呼び出す関数・クラス・メソッド|目的|
|---|---|---|
|`ai_data_scientist.project_manager`|`resolve_project`, `ensure_notebook`, `ensure_data_dir`, `enqueue_write`|指定slug/絶対rootの解決、データ保存先、ソース・根拠・メタデータの直列化書込み|
|`ai_data_scientist.language_router`|`detect_language`|日本語の選択|
|`ai_data_scientist.lifecycle`|`register_run`, `mark_execution_start/end`, `mark_write_start/end`, `is_cancel_requested`, `mark_completed`, `mark_failed`, `get_run_status`, `wait_for_quiescence`|長時間処理前の登録、実行/書込み記録、終了・失敗・静止確認|
|`ai_data_scientist.ingestion`|`SourceSpec`, `ingest`|取得済CSVの読み込みと行数制限|
|`ai_data_scientist.eda`|`explore`|型・欠測・カテゴリ・要約確認|
|`ai_data_scientist.cleaning`|`clean_dataset`|完全重複検査。除去0行を確認|
|`ai_data_scientist.data_definition`|`FieldValue`, `build_manifest`, `DataDefinitionManifest.unresolved_fields`|定義、来歴、unknown/inferredの区別（DataDefinitionManifestは戻り値型であり直接importしていない）|
|`ai_data_scientist.analysis_assumptions`|`Assumption`, `AnalysisAssumptionManifest`, `check_manifest`|記述的範囲・仮定・未解決リスク|
|`ai_data_scientist.data_quality`|`detect_anomalies`|許容値・範囲・欠測の意味的検査|
|`ai_data_scientist.sensitivity`|`SensitivityPlan`, `run_sensitivity`|36仕様と明示的20%判定|
|`ai_data_scientist.visualization`|`render_chart`, `build_image_output`|標準図の実生成とPNG MIME保存。複数パネル図はMatplotlibで実生成|
|`ai_data_scientist.insight_engine`|`extract_cited_value`, `record_insight`; `EvidenceMissingError`を型参照|実行出力から引用値抽出、根拠付Insight保存、stdout不対応の最小再現|
|`ai_data_scientist.notebook_audit`|`audit_notebook(..., visual_audit=True)`|実行・根拠・図表監査、監査欠落の最小再現|

`mcp_gateway`、`stats_analysis`、`dataset_validation`、統計的z-score異常検査は直接import/呼び出していない。不適用理由は分析計画節に記載。Kaggle SDKの`KaggleApi.authenticate`, `dataset_list`, `dataset_list_files`, `dataset_download_file`, `dataset_metadata`、SciPy、Pandas、Matplotlib、nbformatは外部ライブラリであってJupytermindモジュールではない。
'''
improvement_md = '''## Jupytermind改善候補

以下2件はJupytermindの関数を最小Notebookに直接適用して再現した機能不足候補。確定修正やGitHub Issue作成は本分析の範囲外。

|分類|再現条件|期待する挙動|実際の挙動|影響|回避策|関係セル・ログ|
|---|---|---|---|---|---|---|
|機能不足：証拠照合の出力形式|execution_count=1のcodeセルにstdout `STREAM_PROBE=7`を保存し`record_insight(...,1,'STREAM_PROBE=7',...)`|実在するstream出力も根拠として照合|`EvidenceMissingError`。MIME dataのtext/plainなら照合可能|printだけの正常な分析根拠を拒否|分析の要点を`display(Markdown(...))`のMIME出力で保存し、その値を抽出|`jupytermind-probes`、`data/jupytermind_probe_log.json`の`stream_evidence`|
|監査カバレッジ不足：chart metadata欠落|十分なPNGを持つ実行済セルにchart metadataを付けず`audit_notebook(..., visual_audit=True)`|欠落した字形・タイトル・軸・凡例の監査情報を通知|`ok=True`, `visual_findings=[]`|監査成功だけではラベルと字形の確認済を保証しない|主Notebookは実際の描画・警告捕捉で確認したchart metadataを明示的に付与し、図を目視確認|`jupytermind-probes`、同ログの`missing_chart_metadata`、主図セル`figures`/`iteration-2`/`iteration-3`|

**切り分け**：原実験サイトのタイムアウト、旧Seaborn文書URLの404は外部ネットワーク/参照先問題。初期の引用符SyntaxError、メタデータ`info`階層の読み違い、広すぎた文献検索は生成した分析コード側の問題で修正済み。Jupyter MCPの対象親ディレクトリ未作成はbootstrapで解消した環境準備問題。Kaggle認証・検索・ファイル取得は成功。Copilot CLI固有またはベンチマークランナーの不具合は確認していない。以上をJupytermind改善候補に混同しない。

図監査のnear-empty判定はPNG圧縮密度のヒューリスティックであり、画像内容の正しさを保証するものではない。Notebook内のPNGを人が読み直し、タイトル・軸・凡例・曲線・支持域表示を確認した。
'''
stop_md = '''## 反復3の結果・反復停止と解釈の限界

反復3では保存時間点数と整列端点の対応を確認し、全曲線平均がcoherenceごとに異なる時間帯を混ぜることを可視化した。共通サポートへの切り取りだけで行数比が変わるため、判断確率として使えない。証拠は`iteration-3`の端点表、要約対比表、図、出力トークンに保存。

3回の反復で、結論に影響する窓/要約/ベースライン、時間依存、不揃い支持域を調べた。残る問題は試行・個体・ニューロンID、反応時間、正答ラベル、試行分母、CSVの加工辞書の欠落であり、このCSVの追加計算では解消できない。追加検定や最適な窓探索は擬似精度を増すため行わない。上限3回に到達し、利用可能データ内で次に価値のある追加分析はないと判断した。

**推定可能**：表示された条件別神経曲線の時間依存の差、指定共通窓での記述的効果量。**推定不能**：coherenceに対する行動反応時間平均/分布、正答率、選択確率、個体/神経集団の信頼区間、因果効果、個別試行の判断閾値や効果発現潜時。

原論文抄録は強い運動信号で判断が速く・正確になったと報告している（DOI:10.1523/JNEUROSCI.22-21-09475.2002）。これは今回のCSVから独立に再推定した結果ではなく、出典からの文脈説明に限る。原実験との完全な加工同一性も未検証。
'''
final_code = '''# 根拠付きInsightの再生成・再実行一致・図表監査・ライフサイクル終了
start('finalization')
from dataclasses import replace
try:
    stored=nbformat.read(handle.notebook_path,as_version=4)
    by_id={c.id:c for c in stored.cells}
    def evidence(cell_id,pattern):
        cell=by_id[cell_id]
        texts=[str(v) for o in cell.get('outputs',[]) for v in o.get('data',{}).values() if isinstance(v,str)]
        cited=insight_engine.extract_cited_value({'output':'\\n'.join(texts)},pattern)
        return cell.execution_count,cited
    # 再実行後にexecution_countが変わるので、古い参照を捨てて実出力から作り直す。
    write(lambda nb: setattr(nb,'cells',[c for c in nb.cells if not c.metadata.get('managed_insight')]))
    insight_specs=[
      ('manifests',r'(REACTION_TIME_AVAILABLE=False)', 'Insight 1（適用範囲）：取得CSVは848行の24条件曲線で、試行反応時間列がない。timeを反応時間とみなさず、判断頻度・正答率も推定しない。独立試行/個体/ニューロン数が不明なため母集団検定・CIを報告しない。'),
      ('initial-analysis',r'PRIMARY_GAP_INTERACTION=([-0-9.]+)', 'Insight 2（主窓に限定した平均差）：dotsの100..280で平均T1−T2差はcoherence 0で'+f'{gap_by_coh.loc[0.0]:.6f}'+'、51.2で'+f'{gap_by_coh.loc[51.2]:.6f}'+'、高低差の差は'+f'{primary_effect:.6f}'+'（CSV発火率単位）。これは特定窓の記述量であり、全時間帯の効果や判断確率ではない。'),
      ('initial-analysis',r'T1_SLOPE_RHO=([-0-9.]+)', 'Insight 3（傾き）：主窓のcoherenceと線形傾きのSpearman記述係数はT1 '+f'{rank_stats["T1"]:.6f}'+'、T2 '+f'{rank_stats["T2"]:.6f}'+'。6設計水準の関連であって独立被験者の相関検定ではない。OLSとTheil–Senで高低coherenceの傾き差の方向は保たれたが、平均レベルの安定性とは別。'),
      ('iteration-1',r'(SENSITIVITY_STABLE=False)', 'Insight 4（反復1・結論修正）：36仕様の効果範囲は'+f'{sensitivity_table.effect.min():.6f}'+'〜'+f'{sensitivity_table.effect.max():.6f}'+'。stable=False、max_relative_deviation='+f'{sensitivity_report.max_relative_deviation:.6f}'+'（許容0.20）。方向も大きさも仕様全体では安定せず、「刺激が強いほど選択差が常に大きい」は撤回する。この範囲はCIではない。'),
      ('iteration-2',r'LATE_CORRECTED_MEAN=([-0-9.]+)', 'Insight 5（反復2・時間依存）：ベースライン補正済み高低coherenceの選択差の差は100..140で'+f'{blocks.iloc[0].corrected_mean:.6f}'+'、260..280で'+f'{blocks.iloc[-1].corrected_mean:.6f}'+'。主窓10点中正なのは4点で、後半の大きい差が平均を押し上げる。探索的ブロックは独立標本でなく、符号交差を反応時間/統計的潜時と呼ばない。'),
      ('iteration-3',r'MATCHED_ROW_T2_SHARE=([-0-9.]+)', 'Insight 6（反復3・支持域）：全行T2比率は'+f'{raw_share:.6f}'+'、共通サポートでは'+f'{matched_share:.6f}'+'に変わる。全12条件でdots終点+sacc始点=120という対応もあり、行数は曲線の保存ビン数を反映する。比率を選択確率、終点を試行RTとして推定する根拠はない。')]
    for cell_id,pattern,text in insight_specs:
        count,cited=evidence(cell_id,pattern)
        lifecycle.mark_write_start(run_id); event('insight','write_start')
        try:
            insight_engine.record_insight(handle,text,count,cited,'descriptive',language='ja')
        finally:
            lifecycle.mark_write_end(run_id); event('insight','write_end')
    def annotate(nb):
        for c in nb.cells:
            if c.cell_type=='markdown' and c.source.startswith('Insight '): c.metadata['managed_insight']=True
            if c.id=='figures':
                c.metadata['chart']={'missing_glyphs':[],'title':'Neural trajectories and common-window contrasts','xlabel':'Relative time / coherence (CSV units)','ylabel':'Firing rate / slope / choice contrast (CSV units)','legend':'Coherence / choice / alignment-window','charts':[chart_specs[n] for n in ['trajectories','matched_contrasts','skill_gap_chart']]}
            elif c.id=='iteration-2': c.metadata['chart']=chart_specs['time_interaction']
            elif c.id=='iteration-3': c.metadata['chart']=chart_specs['support_diagnostic']
        footer=next(c for c in nb.cells if c.id=='final-audit')
        nb.cells=[c for c in nb.cells if c.id!='final-audit']+[footer]
        nb.metadata['run_id']=run_id
        nb.metadata['provenance']=provenance
        nb.metadata['data_definition_manifest']=asdict(definition)
        nb.metadata['source_verification']={k:{kk:vv for kk,vv in v.items() if kk!='text'} for k,v in source_docs.items()}
    write(annotate)
    assumptions=replace(assumptions,assumptions=tuple(replace(a,statement='36仕様で窓・要約・補正依存を検証。安定と仮定せず時間別記述に限定',status='tested',evidence_cell=by_id['iteration-1'].execution_count) if a.id=='window_choice' else a for a in assumptions.assumptions))
    final_findings=check_manifest(assumptions)
    print('最終分析前提の指摘（残存リスクをすべて提示）:',[asdict(f) for f in final_findings])
    print('単位・T1/T2写像・平滑化/集約方法は未確認。CSV数値単位の記述に限定する。')
    (data_dir/'final_analysis_assumptions_manifest.json').write_text(json.dumps(asdict(assumptions),ensure_ascii=False,indent=2),encoding='utf-8')
    reference=json.loads((data_dir/'first_pass_reference.json').read_text(encoding='utf-8'))
    actual={'primary_effect':primary_effect,'sensitivity_min':float(sensitivity_table.effect.min()),'sensitivity_max':float(sensitivity_table.effect.max()),'max_relative_deviation':sensitivity_report.max_relative_deviation,'early_corrected_mean':float(blocks.iloc[0].corrected_mean),'late_corrected_mean':float(blocks.iloc[-1].corrected_mean),'raw_row_share':raw_share,'matched_row_share':matched_share}
    checks={k:bool(np.isclose(v,reference[k],rtol=1e-10,atol=1e-10)) for k,v in actual.items()}
    checks['same_sha256']=provenance['sha256']==reference['sha256']
    print('初回数値・SHA-256との一致:',checks)
    assert all(checks.values()), '初回と再実行で実質的に結果が変化しました'
    (data_dir/'reexecution_comparison.json').write_text(json.dumps({'reference':reference,'actual':actual,'checks':checks},ensure_ascii=False,indent=2),encoding='utf-8')
    report=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
    print('notebook_audit:',json.dumps(asdict(report),ensure_ascii=False,indent=2))
    print('audit_ok:',report.ok)
    if not report.ok: raise RuntimeError('Notebook監査に未解決の指摘があります')
    write(lambda nb: nb.metadata.update({'notebook_audit':asdict(report),'notebook_audit_ok':report.ok,'analysis_assumptions_manifest':asdict(assumptions),'analysis_assumption_findings':[asdict(f) for f in final_findings],'reexecution_checks':checks,'jupytermind_improvement_candidates':probe_results,'iteration_count':3}))
except Exception:
    lifecycle.mark_failed(run_id)
    raise
finally:
    end('finalization')
lifecycle.mark_completed(run_id)
status=lifecycle.wait_for_quiescence(run_id,timeout_s=10)
assert status.state=='completed' and status.active_cell_executions==0 and status.pending_notebook_writes==0 and status.locks_held==0
write(lambda nb: nb.metadata.update({'lifecycle':asdict(status),'phase_log':phase_log}))
(data_dir/'lifecycle.json').write_text(json.dumps({'status':asdict(status),'phase_log':phase_log},ensure_ascii=False,indent=2),encoding='utf-8')
print('最終ライフサイクル:',asdict(lifecycle.get_run_status(run_id)))
display(Markdown('FINAL_STATUS=completed; REEXECUTION_NUMERIC_MATCH=True; NOTEBOOK_AUDIT_OK=True; VISUAL_AUDIT=True'))
'''
add_cells(md(stop_md),md(module_md),md(improvement_md),md('## 再実行・監査\n\n初回結果は`data/first_pass_reference.json`に固定した。カーネルを再起動し、全コードセルを上からJupyter MCPで再実行する。最終セルは最新出力からInsight根拠を再生成し、SHA-256と主要数値の一致を確認する。実行中の最後の自己監査セルは監査規約に従い未実行扱いから除外され、終了後に保存済Notebookをもう一度外側のJupyter MCPセルで読み取り監査する。結果はNotebook metadataと`data/notebook_audit_postpersist.json`に保存。'),code(final_code,'final-audit'))
# 再実行時も失敗状態を明示し、前フェーズの実行カウンタが残らないようにする。
def add_failure_hook(nb):
    setup=next(c for c in nb.cells if c.id=='setup')
    setup.source += '''\n# セル失敗時にライフサイクルをfailedとして明示的に記録する。\ndef _record_cell_failure(result):\n    failure = result.error_before_exec or result.error_in_exec\n    if failure is not None:\n        lifecycle.mark_failed(run_id)\n        event('cell_failure',type(failure).__name__)\n        while lifecycle.get_run_status(run_id).active_cell_executions:\n            lifecycle.mark_execution_end(run_id)\n        (data_dir/'lifecycle_failure.json').write_text(json.dumps({'status':asdict(lifecycle.get_run_status(run_id)),'events':phase_log},ensure_ascii=False,indent=2),encoding='utf-8')\nif '_registered_failure_hook' in globals():\n    get_ipython().events.unregister('post_run_cell',_registered_failure_hook)\n_registered_failure_hook=_record_cell_failure\nget_ipython().events.register('post_run_cell',_registered_failure_hook)\n'''
    for c in nb.cells:
        if c.cell_type=='code': compile(c.source,f'<{c.id}>','exec')
    nb.metadata['first_pass_reference']=first_pass
    nb.metadata['recovered_preparation_issues']=[{'cell':'source-verification','classification':'analysis_code','error':'SyntaxError','resolution':'引用符構成を修正して実行成功'},{'cell':'source-verification','classification':'external_source','error':'URLError timeout / old documentation HTTPError','resolution':'原論文DOIでEurope PMC照合。未確認定義はinferred/unknownのまま'}]
write(add_failure_hook)
print({'prepared_code_cells':len([c for c in nbformat.read(handle.notebook_path,as_version=4).cells if c.cell_type=='code']),'first_pass_sha256':first_pass['sha256']})

{'prepared_code_cells': 12, 'first_pass_sha256': 'dd8ed5e18358ec23250ecc877c98d0212c0419d0152e1bf5387b20341059842b'}


In [13]:
# 全セル保存後の外側Jupyter MCP監査。対象Notebookのコードは実行しない。
postpersist=notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True)
assert postpersist.ok and postpersist.executed_code_cell_count==12 and not postpersist.findings and not postpersist.visual_findings
persisted=nbformat.read(handle.notebook_path,as_version=4)
code_counts=[c.execution_count for c in persisted.cells if c.cell_type=='code']
assert code_counts==list(range(1,13)), '再起動後の上から順の全セル実行を確認できません'
assert persisted.metadata['lifecycle']['state']=='completed'
assert all(persisted.metadata['reexecution_checks'].values())
(data_dir/'notebook_audit_postpersist.json').write_text(json.dumps({'report':asdict(postpersist),'ok':postpersist.ok,'execution_counts':code_counts,'lifecycle':asdict(lifecycle.get_run_status(run_id))},ensure_ascii=False,indent=2),encoding='utf-8')
def record_postpersist(nb):
    nb.metadata['notebook_audit_postpersist']=asdict(postpersist)
    nb.metadata['notebook_audit_postpersist_ok']=postpersist.ok
    nb.metadata['reexecution']={'mode':'Jupyter MCP execute_cell, restarted kernel, top-to-bottom','execution_counts':code_counts,'all_code_cells':12,'sha256_match':True,'numeric_match':True,'completed_at_utc':datetime.now(timezone.utc).isoformat()}
    for c in nb.cells:
        if c.cell_type=='markdown' and c.source.startswith('## 使用したJupytermindモジュール'):
            c.source=c.source.replace('`mark_completed`, `mark_failed`, `get_run_status`','`mark_completed`, `get_run_status`')
            c.source += '\n\n`lifecycle.mark_failed`は失敗時フック/例外経路に直接呼び出すよう定義したが、最終の成功した全セル再実行ではその経路は実行されていないため、成功実行の使用済み関数一覧には含めない。'
        if c.cell_type=='markdown' and c.source.startswith('## 再実行・監査'):
            c.source += '\n\n**保存後の最終結果**：全12コードセルのexecution_countは1..12。notebook_auditはnbformat_valid=True、executed=12/12、未実行0、エラー0、根拠付きInsight6、画像5（3図セル）、findings=[]、visual_findings=[]、ok=True。主要数値8項目とファイルSHA-256は初回一致。lifecycleはcompleted、active executions/pending writes/locksはすべて0。再現検査したJupytermind改善候補2件と未検証の物理単位は別途残る。'
write(record_postpersist)
# 書込み後も監査値が維持されることを確認する。
assert notebook_audit.audit_notebook(handle.notebook_path,visual_audit=True).ok
print({'audit_ok':postpersist.ok,'executed':postpersist.executed_code_cell_count,'code_counts':code_counts,'insights':postpersist.insight_cell_count,'visual_findings':len(postpersist.visual_findings),'run':asdict(lifecycle.get_run_status(run_id)),'notebook_bytes':handle.notebook_path.stat().st_size,'bootstrap_exists':(repo/'dots-bootstrap-v020-exp-15.ipynb').exists()})

AssertionError: 